In [5]:
import requests
from typing import List, Dict, Optional


In [6]:

def get_supermarkets_near(
    lat: float,
    lon: float,
    radius_meters: int = 2000,
    chains: Optional[List[str]] = None
) -> List[Dict]:
    """
    Get supermarkets near given coordinates using Overpass API.
    
    Args:
        lat: Latitude of center point
        lon: Longitude of center point
        radius_meters: Search radius in meters (default: 2000m = 2km)
        chains: Optional list of chain names to filter (e.g. ['REWE', 'Edeka'])
                If None, returns all supermarkets
    
    Returns:
        List of dicts with supermarket info (name, lat, lon, tags)
    
    Example:
        # All supermarkets within 3km
        markets = get_supermarkets_near(51.3406, 7.0453, 3000)
        
        # Only specific chains within 2km
        markets = get_supermarkets_near(51.3406, 7.0453, 2000, ['REWE', 'Edeka', 'Aldi'])
    """
    
    overpass_url = "http://overpass-api.de/api/interpreter"
    
    # Build chain filter if provided
    chain_filter = ""
    if chains:
        chain_pattern = "|".join(chains)
        chain_filter = f'["name"~"^({chain_pattern})$",i]'
    
    # Overpass QL query
    overpass_query = f"""
    [out:json];
    (
      node["shop"="supermarket"]{chain_filter}(around:{radius_meters},{lat},{lon});
      way["shop"="supermarket"]{chain_filter}(around:{radius_meters},{lat},{lon});
    );
    out center tags;
    """
    
    try:
        response = requests.get(overpass_url, params={'data': overpass_query}, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        results = []
        for element in data.get('elements', []):
            # Get coordinates (nodes have lat/lon, ways have center)
            if element['type'] == 'node':
                element_lat = element['lat']
                element_lon = element['lon']
            else:  # way
                element_lat = element.get('center', {}).get('lat')
                element_lon = element.get('center', {}).get('lon')
            
            # Extract useful info
            tags = element.get('tags', {})
            results.append({
                'name': tags.get('name', 'Unknown'),
                'lat': element_lat,
                'lon': element_lon,
                'street': tags.get('addr:street'),
                'housenumber': tags.get('addr:housenumber'),
                'postcode': tags.get('addr:postcode'),
                'city': tags.get('addr:city'),
                'opening_hours': tags.get('opening_hours'),
                'phone': tags.get('phone'),
                'website': tags.get('website'),
                'brand': tags.get('brand'),
                'all_tags': tags  # Full OSM tags if you need more
            })
        
        return results
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return []


# Usage examples:
if __name__ == "__main__":

    markets = get_supermarkets_near(51.3406, 7.0453, 2000)
    print(f"Found {len(markets)} supermarkets")
    for m in markets:
        print(f"- {m['name']} at {m['street']} {m['housenumber']}, {m['postcode']} {m['city']}, {m['lat']} {m['lon']}")
    


Found 16 supermarkets
- EDEKA Mader at Röntgenstraße 11, 42549 Velbert, 51.3394133 7.0197516
- ALDI Nord at Heiligenhauser Straße 60, 42549 Velbert, 51.3327009 7.0223837
- REWE at None None, None None, 51.3314691 7.0363461
- Lidl at Güterstraße 11, 42551 None, 51.341866 7.0519297
- Lidl at None None, None None, 51.3314886 7.0214912
- Eliz Center at None None, None None, 51.3415222 7.0412964
- ALDI Nord at Friedrichstraße 305, 42551 Velbert, 51.3313019 7.0556813
- denn's Biomarkt at Heiligenhauser Straße 56, None Velbert, 51.3324716 7.0226007
- Reformhaus Vita Nova Kaubisch at None None, None None, 51.3409019 7.0434672
- Kaufland at None None, None Velbert, 51.3356001 7.049484
- PENNY at Schloßstraße 65, 42551 Velbert, 51.3462305 7.044966
- EDEKA Hundrieser at Sontumer Straße 73, 42551 Velbert, 51.3308889 7.0587972
- Al Essa Markt at None None, None None, 51.3412319 7.0478241
- Netto Marken-Discount at Kolpingstraße 32, None None, 51.3417611 7.0458949
- EUFOOMA at Heiligenhauser Straße 